# GPT-style Decoder-only Transformer — 직접 학습시켜보기 (Colab GPU용)

**사용법**
1. 런타임 → 런타임 유형 변경 → **GPU(T4)** 선택
2. 위에서 아래로 셀을 순서대로 실행 (torch는 Colab에 기본 설치되어 있음)
3. 학습이 끝나면 마지막 셀에서 `generate()`로 문장을 직접 생성

지금까지 numpy로 구현했던 것과 동일한 개념(Q/K/V, Masked Self-Attention, Add&Norm,
FeedForward, Positional Embedding)을 PyTorch로 옮기고, 실제로 학습이 되도록
autograd(analytic backprop)와 GPU 연산을 사용합니다.

**구조**: GPT는 "Decoder만" 사용하는 모델입니다.
- Masked Self-Attention만 있고, Encoder나 Encoder-Decoder(Cross) Attention은 없음
- "지금까지 나온 텍스트만 보고 다음 글자를 예측"하는 순수 언어모델(Language Model)


## 0. 환경 설정

In [ ]:
import math
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

torch.manual_seed(42)


## 1. 데이터 준비 (tiny-shakespeare, 글자 단위 토큰화)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(DATA_URL, "input.txt")

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))          # 등장하는 모든 글자 (알파벳, 공백, 문장부호 등)
vocab_size = len(chars)
print(f"vocab_size: {vocab_size}  (문자 목록: {''.join(chars[:20])}...)")

stoi = {ch: i for i, ch in enumerate(chars)}   # 글자 -> id
itos = {i: ch for i, ch in enumerate(chars)}   # id -> 글자
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]      # 90% 학습 / 10% 검증


## 2. 하이퍼파라미터 (무료 Colab T4 GPU 기준, ~10분 내 유의미한 결과)

In [ ]:
block_size = 128      # 한 번에 보는 문맥 길이 (앞서 배운 seq_len)
batch_size = 64        # 한 번에 몇 문장을 병렬로 학습할지
d_model = 256           # 임베딩/hidden 차원
num_heads = 8            # Multi-Head Attention의 헤드 수
num_layers = 6            # Decoder Layer를 몇 겹 쌓을지 (그림의 "N×")
d_ff = 4 * d_model         # FeedForward hidden 차원 (관례적으로 d_model의 4배)
dropout = 0.1
learning_rate = 3e-4
max_iters = 3000            # 학습 스텝 수 (필요하면 늘려도 됨)
eval_interval = 300
eval_iters = 50


In [ ]:
def get_batch(split):
    """랜덤한 위치에서 block_size만큼 잘라 (입력, 정답) 쌍을 batch_size개 만든다."""
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])   # 한 칸 밀린 다음 글자가 정답
    return x.to(device), y.to(device)


## 3. 모델 정의 (Masked Multi-Head Self-Attention 기반 Decoder-only)

### 3-1. Multi-Head Self-Attention

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """지금까지 손으로 계산했던 Q, K, V + Masking + Softmax + concat + Linear를
       PyTorch의 autograd가 자동으로 미분 가능하도록 nn.Module로 구현"""

    def __init__(self, d_model, num_heads, block_size, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # 미래 토큰을 못 보게 하는 causal mask (register_buffer: 학습 안 되는 고정값)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("causal_mask", mask)

    def forward(self, x):
        B, T, C = x.shape   # Batch, seq_len, d_model

        Q = self.Wq(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)  # (B, heads, T, d_k)
        K = self.Wk(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.Wv(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)             # QK^T / sqrt(d_k)
        scores = scores.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))  # masking
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)

        out = weights @ V                                                      # 가중합
        out = out.transpose(1, 2).contiguous().view(B, T, C)                    # concat
        return self.resid_dropout(self.Wo(out))                                  # Linear


### 3-2. FeedForward

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


### 3-3. Decoder Block (Pre-LN: LayerNorm을 먼저 적용)

In [ ]:
class DecoderBlock(nn.Module):
    """Masked Self-Attention + FeedForward, 각각 Add&Norm (Pre-LN 방식: LayerNorm을 먼저 적용)"""

    def __init__(self, d_model, num_heads, d_ff, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # Residual Connection ("gradient highway")
        x = x + self.ffn(self.ln2(x))
        return x


### 3-4. GPT (전체 모델 조립)

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)     # Input Embedding
        self.position_embedding = nn.Embedding(block_size, d_model)  # Learned Positional Embedding

        self.blocks = nn.Sequential(
            *[DecoderBlock(d_model, num_heads, d_ff, block_size, dropout) for _ in range(num_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.lm_head.weight = self.token_embedding.weight   # Weight Tying (임베딩-출력층 공유)

        self.apply(self._init_weights)   # 학습 초반부터 안정적으로 시작하도록 가중치 초기화

    def _init_weights(self, module):
        """GPT류 모델들이 흔히 쓰는 초기화 (표준편차 0.02짜리 정규분포).
        초기 로짓 크기를 작게 유지해서, 학습 시작 시 Loss가 ln(vocab_size) 근처에서
        출발하도록 만든다 (기본 PyTorch 초기화를 쓰면 초반 Loss가 불필요하게 크게 튐)."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)                             # (B,T,d_model)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb                                             # Embedding + Positional
        x = self.blocks(x)                                                 # Decoder Layer x N
        x = self.ln_f(x)
        logits = self.lm_head(x)                                            # Linear (Weight Tying)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        """학습된 모델로 글자를 하나씩 이어서 생성 (autoregressive generation)"""
        self.eval()   # dropout 등을 비활성화 (no_grad는 gradient만 끄고 dropout은 안 끔)
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]           # 최근 block_size 글자만 문맥으로 사용
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]                        # 마지막 위치의 예측만 사용
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)  # 확률적으로 다음 글자 샘플링
            idx = torch.cat([idx, next_id], dim=1)
        self.train()   # 이후 계속 학습을 이어갈 수도 있으므로 원상복귀
        return idx


## 4. 학습 루프

In [ ]:
model = GPT(vocab_size, d_model, num_heads, num_layers, d_ff, block_size, dropout).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"모델 파라미터 수: {num_params / 1e6:.2f}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


In [ ]:
print("\n" + "=" * 60)
print("학습 시작")
print("=" * 60)

for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step:5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()          # <- 지금까지 손으로 계산했던 chain rule을 PyTorch가 analytic하게 자동 수행
    optimizer.step()          # <- w_new = w_old - lr * gradient (경사하강법)


## 5. 학습된 모델로 문장 생성해보기

In [ ]:
print("\n" + "=" * 60)
print("생성 결과")
print("=" * 60)

context = torch.zeros((1, 1), dtype=torch.long, device=device)   # 빈 문맥에서 시작
generated = model.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))


In [ ]:
# 체크포인트 저장 (Colab에서 다운로드해서 나중에 이어서 쓰고 싶을 때)
torch.save(model.state_dict(), "gpt_shakespeare.pt")
print("\n모델 저장 완료: gpt_shakespeare.pt")
